# Dzyaloshinskii-Moriya Interaction (DMI) Model

The Dzyaloshinskii-Moriya Interaction (DMI) model describes an
antisymmetric exchange interaction between neighboring spins. This
interaction can arise from spin-orbit coupling and produces chiral
spin dynamics between two qubits.

The DMI Hamiltonian is given by:

$$
H_{\mathrm{DMI}}
=
D\left(
S_1^x S_2^y
-
S_1^y S_2^x
\right)
$$

where $D$ represents the strength of the Dzyaloshinskii-Moriya
interaction. Unlike symmetric spin interactions, the DMI term couples
orthogonal spin components and introduces a directional phase structure
into the quantum evolution.

Using the Pauli spin representation:

$$
S^\alpha = \frac{1}{2}\sigma^\alpha
$$

the Hamiltonian can be expressed in terms of qubit Pauli operators as:

$$
H_{\mathrm{DMI}}
=
\frac{D}{4}
\left(
X\otimes Y
-
Y\otimes X
\right)
$$

The competition between the $X\otimes Y$ and $Y\otimes X$ terms produces
chiral and phase-dependent dynamics. This makes the evolution sensitive
to the relative orientation of the two spins and can generate asymmetric
population transfer and non-trivial quantum coherence between the qubits.

### PAULI-BASIS MEASUREMENTS for DMI Model

In [ ]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_fez")

shots = # type in the number of shots depending up on your problem

iterations = # number of Floquet steps

# DMI parameter
D =

print("Backend:", backend.name)
print("DMI steps:", iterations)

# DMI 2-QUBIT MODEL CIRCUIT
def dmi_circuit(n, D):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.h(1)

    for _ in range(n):

        qc.sdg(1)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * D, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(1)

        qc.sdg(0)
        qc.h(0)
        qc.h(1)
        qc.cx(0, 1)
        qc.rz(2 * D, 1)
        qc.cx(0, 1)
        qc.h(0)
        qc.h(1)
        qc.s(0)

    return qc

# BASIS ROTATIONS + MEASUREMENT
def add_basis_and_measure(qc, basis):
    qc2 = qc.copy()
    for q, b in enumerate(basis):
        if b == "X":
            qc2.h(q)
        elif b == "Y":
            qc2.sdg(q)
            qc2.h(q)
    qc2.measure_all()
    return qc2

bases = ["ZZ","ZX","ZY","ZI",
         "XZ","XX","XY","XI",
         "YZ","YX","YY","YI",
         "IZ","IX","IY"]

# BUILD CIRCUITS (FLOQUET STEP SWEEP)
all_circuits = []

for n in range(1, iterations + 1):
    for basis in bases:
        base = dmi_circuit(n, D)
        full = add_basis_and_measure(base, basis)
        all_circuits.append(full)

print("Total circuits submitted:", len(all_circuits))  # 75 Circuits

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(all_circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits, shots=shots)

print("\nSAVE THIS JOB ID:")
print(job.job_id())

## Retrieving the Job Results and Performing State Analysis

After submitting the quantum circuits to the IBM Quantum backend, the Job ID
printed by the execution code should be saved. The Job ID uniquely identifies
the submitted experiment and allows the experimental results to be retrieved
later without executing the quantum circuits again.

The saved Job ID is entered into the following analysis code:

```python
job = service.job("YOUR_JOB_ID")